# RecVis: PE-Core-G zero-shot y vistas locales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/pe_semantic.ipynb)

**Experimento nuevo, sin puntuación Kaggle todavía.** El experimento anterior obtuvo 94.015% con SigLIP semántico y 94.844% con el ensemble (resultados comunicados por el participante). Aquí probamos otro modelo de imagen y texto, **PE-Core-G14-448**, con aproximadamente 1.88B parámetros visuales y 0.47B textuales.

Este notebook prepara resultados para cuando vuelvan a estar disponibles las entregas. No envía nada a Kaggle, no modifica el notebook anterior y lee sus probabilidades desde Drive. Ejecútalo en una sesión separada cuando puedas asignarle la GPU.

**Hipótesis:** otro preentrenamiento imagen-texto puede aportar errores distintos de SigLIP. Una vista algo más cercana puede ayudar con objetos representados dentro de un soporte. Ninguna de las dos mejoras está demostrada en este dataset.

| Archivo | Método |
|---|---|
| `submission_pe_zero_shot.csv` | PE, imagen completa + flip, promedio de logits |
| `submission_pe_local_views.csv` | PE con vistas locales adicionales en las imágenes de menor margen |
| `submission_siglip_pe_zero_shot.csv` | 50% probabilidades SigLIP anterior + 50% PE de imagen completa/flip |

El 50/50 es una comparación inicial fija, no un óptimo. El ensemble SigLIP+PE utiliza PE sin vistas locales para separar el cambio de arquitectura del cambio de vistas. No se añaden las cabezas supervisadas ni DINO a esta nueva combinación.

## 1. Dependencias y código

Selecciona A100. Se conservan PyTorch y torchvision de Colab. Se descargan únicamente los archivos del encoder PE y su tokenizador desde una revisión oficial fija, verificando sus hashes; no se instala el entorno completo de vídeo/PLM.

Si esta sesión había importado otra versión de Transformers, reiníciala después de instalar. PE es público: **este experimento no requiere el token gated de DINO**.

In [ ]:
%pip -q install "transformers==4.57.3" "huggingface_hub>=0.34,<1" "timm==1.0.15" "einops==0.8.1" "ftfy==6.3.1" regex kagglehub

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, sys
import requests

SOURCES = {
    'semantic_vit': ('fc538b90a48c2c1ce7cd8f3f96045653478744e9', '8d3baa53881282e34cbdb21c3527c7164213f3e78c1d921f393193c011958afc'),
    'pe_semantic': ('c30fc5422f23728a42cdc18741dac7ef52a7154c', '00f6ede2beb99c5abe82e3a6813e06f1802d2dedca3b82576fb2477ce951c214'),
}
CODE_DIR = Path('/content/recvis_pe_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, (revision, expected_hash) in SOURCES.items():
    url = f'https://raw.githubusercontent.com/rivas-vega/A3_Recv/{revision}/{name}.py'
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    if hashlib.sha256(response.content).hexdigest() != expected_hash:
        raise RuntimeError(f'El archivo {name} no coincide con la revisión fijada.')
    path = CODE_DIR/f'{name}.py'
    path.write_bytes(response.content)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import semantic_vit as sv
import pe_semantic as pe
print('Código verificado:', SOURCES)

## 2. Configuración

`PREVIOUS_RUN_NAME` debe apuntar a la carpeta que contiene `siglip_zero_shot_probabilities.npz` del experimento anterior. No basta con tener su CSV: necesitamos las probabilidades. Si no están disponibles, PE puede producir sus dos CSV por separado.

Las vistas locales son cinco recortes de **28×28 sobre los 32×32 originales**, después ampliados por el procesador oficial a 448×448. Solo se calculan para el **25% con menor margen entre las dos primeras clases**, sin consultar etiquetas. En esas imágenes se conserva el 75% del logit original y se añade un 25% del promedio de los recortes. Son parámetros fijados antes de ver los resultados de PE en Kaggle.

Mantén el nuevo `RUN_NAME` para reanudar. Cambia ese nombre si cambias parámetros. Las vistas guardadas se reutilizan; no se sobrescriben resultados anteriores.

In [ ]:
from google.colab import drive
import torch

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Selecciona una A100 o una GPU compatible con BF16.')
print(torch.cuda.get_device_name(), round(torch.cuda.get_device_properties(0).total_memory/2**30,1), 'GiB VRAM')
torch.manual_seed(42)
torch.set_float32_matmul_precision('high')
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/A3_Recv')
PREVIOUS_RUN_NAME = 'vit_giant_7b_v1'
RUN_NAME = 'pe_g14_448_v1'
RUN_DIR = BASE/'runs'/RUN_NAME
SETTINGS = dict(seed=42, previous_run=str(BASE/'runs'/PREVIOUS_RUN_NAME),
                crop_size=28, uncertain_fraction=.25, crop_weight=.25,
                run_local_views=True, run_stress_audit=True)
SESSION_BUDGET_HOURS = 7.0
budget = sv.Budget(SESSION_BUDGET_HOURS)
print('Nuevo experimento:', RUN_DIR)
print('Probabilidades SigLIP anteriores:', SETTINGS['previous_run'])

## 3. Datos

Usa tu cuenta Kaggle con las reglas aceptadas. El código lee imágenes y etiquetas de train/validación, y solamente imágenes e IDs de test. No se usa train para ajustar pesos: unas imágenes sirven para medir el rendimiento de la GPU. Las etiquetas de validación se utilizan para calibrar temperaturas y evaluar.

In [ ]:
import kagglehub
from google.colab import userdata
if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
    del token
kagglehub.login()
DATA_ROOT = Path(kagglehub.competition_download('mva-recvis-2026'))
data = sv.read_data(DATA_ROOT)
experiment = pe.PEExperiment(data, SETTINGS, RUN_DIR, BASE/'cache_pe_semantic',
                             Path('/content/recvis_pe_chunks'), budget)
sv.atomic_json(dict(sources=SOURCES, pe_upstream_revision=pe.PE_REVISION),RUN_DIR/'code_revision.json')
for split in ('train','val','test'):
    print(split, data[split].shape)
previous = Path(SETTINGS['previous_run'])/'siglip_zero_shot_probabilities.npz'
print('Probabilidades SigLIP encontradas:', previous.exists())

## 4. Extracción zero-shot y vistas locales

Se cargan estrictamente los pesos publicados: no se acepta ningún parámetro ausente. Se usan las mismas 80 plantillas públicas de OpenCLIP y los diez nombres CIFAR, sin generar textos nuevos. El modelo permanece congelado.

El procesado es el **oficial de PE**: PIL, bilinear, imagen completa a 448×448 y media/desviación 0.5. Los pesos se conservan en FP32; las operaciones matriciales se ejecutan con autocast BF16. La A100 se utiliza para la inferencia, y el benchmark incluye el coste del preprocesado. Se mide el batch adecuado, memoria y rendimiento antes de estimar las pasadas largas.

Primero se guarda el CSV PE completo y su combinación con SigLIP. Después se calculan los recortes. El presupuesto se controla entre bloques; las descargas y las latencias de Drive no tienen una duración garantizada. Los resultados completos quedan disponibles si la sesión se interrumpe.

La calibración utiliza el mismo 20% reservado de validación que el experimento anterior; se informa la accuracy sobre el 60% reservado. El porcentaje restante no ajusta ningún peso en este notebook. Los índices de menor margen se seleccionan sin etiquetas. Ninguna métrica local se presenta como accuracy del test sintético.

In [ ]:
experiment.run()
print('PE terminado. Resultados:', RUN_DIR)

## 5. Comparación local y diagnóstico de datos

Se compara PE con y sin vistas locales en la misma validación. Un diagnóstico adicional usa hasta 1,000 imágenes reservadas, con gris, posterización a cuatro bits y desenfoque leve, manteniendo sus etiquetas originales. Sirve para medir sensibilidad a cambios de apariencia; **no reproduce el dominio de juguetes, tatuajes o cómics** y no decide automáticamente qué CSV es mejor en Kaggle.

La ampliación por interpolación no crea detalles reales. Los recortes pueden eliminar parte del objeto; por eso son suaves, limitados a un subconjunto y conservan el mayor peso en la imagen completa. Se guarda el candidato sin recortes como control.

In [ ]:
import pandas as pd
from IPython.display import display

rows = [json.loads(p.read_text()) for p in sorted(RUN_DIR.glob('*_metrics.json'))]
if rows:
    frame = pd.DataFrame(rows)
    display(frame[[c for c in ('name','accuracy','nll','temperature','kaggle_public_accuracy') if c in frame]])
stress = RUN_DIR/'pe_stress_audit.json'
if stress.exists():
    report = json.loads(stress.read_text())
    display(pd.DataFrame(report['rows'])[['mode','rows','accuracy']])
    print(report['limitation'])
for path in sorted(RUN_DIR.glob('submission_*.csv')):
    print(path.name)

## 6. Descargar, sin enviar a Kaggle

Esta celda también funciona si la fase de recortes se interrumpió después del primer CSV. El ZIP contiene resultados y diagnósticos, no pesos ni datos privados. Conserva el ensemble anterior de 94.844% como referencia. Cuando puedas enviar de nuevo, PE solo permite evaluar el cambio de modelo; PE con vistas locales permite evaluar el cambio de vistas. La mezcla SigLIP+PE es otro candidato, no un ganador garantizado.

In [ ]:
from google.colab import files
files.download(str(experiment.bundle()))

## Fuentes y reproducibilidad

- [PE-Core-G14-448: modelo y tamaños](https://huggingface.co/facebook/PE-Core-G14-448).
- [Implementación oficial fijada](https://github.com/facebookresearch/perception_models/tree/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder), incluidos su transform y tokenizador. Se descarga y conserva `LICENSE.PE`.
- [Plantillas OpenCLIP fijadas](https://github.com/mlfoundations/open_clip/blob/4cc208d3a44a2fb6d6435006a46f520d2ccc29b1/src/open_clip/zero_shot_metadata.py).

Se guardan hashes de datos, código, revisiones de pesos, versiones, características por bloques, probabilidades y alineación de IDs. Las cachés distinguen cada recorte, subconjunto y transformación. Los parámetros 25% / 28px / 25% y el ensemble 50/50 son decisiones iniciales explícitas, sin ajuste a puntuaciones Kaggle de PE.

Validación de desarrollo: tests sintéticos CPU de los contratos de caché, recortes, selección por incertidumbre, mezcla, lectura de predicciones anteriores y carga estricta. El checkpoint completo y el tiempo A100 deben verificarse al ejecutar este notebook.